In [ ]:
# ===== CONFIGURE HERE (Change based on your setup) =====
# For Google Colab:
# BASE_URL = "/content/drive/MyDrive/experiments"

# For Local (Windows):
BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"

# For Local (Mac/Linux):
# BASE_URL = "/Users/yourname/experiments"

from pathlib import Path
BASE_PATH = Path(BASE_URL)
ARCADE_PATH = BASE_PATH / 'datasets' / 'ARCADE'

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

In [ ]:
# ===== CONFIGURE HERE (Change based on your setup) =====
# For Google Colab:
# BASE_URL = "BASE_URL/experiments"

# For Local (Windows):
BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"

# For Local (Mac/Linux):
# BASE_URL = "/Users/yourname/experiments"

from pathlib import Path
BASE_PATH = Path(BASE_URL)
ARCADE_PATH = BASE_PATH / 'datasets' / 'ARCADE'
RESULTS_PATH = BASE_PATH / 'results'

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

print(f'✓ Base path: {BASE_PATH}')
print(f'✓ Dataset path: {ARCADE_PATH}')

# Simple U-Net Segmentation for ARCADE Dataset

**For Google Colab & Local Training**

Learn how vessel segmentation works with a simple U-Net model.

🎯 Goals:
- Load ARCADE coronary angiography images
- Build simple U-Net architecture
- Train with Dice loss
- Evaluate performance
- Visualize predictions

## Setup: Configure Paths

⚠️ **IMPORTANT:** Modify `BASE_URL` based on where you put the experiments folder!

In [ ]:
# ===== CONFIGURE HERE (Change based on your setup) =====
# For Google Colab:
# BASE_URL = "BASE_URL/experiments"

# For Local (Windows):
BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"

# For Local (Mac/Linux):
# BASE_URL = "/Users/yourname/experiments"

# ======================================================

from pathlib import Path
BASE_PATH = Path(BASE_URL)
ARCADE_PATH = BASE_PATH / "datasets" / "ARCADE"
RESULTS_PATH = BASE_PATH / "results" / "1_simple_unet"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Base path: {BASE_PATH}")
print(f"✓ Dataset path: {ARCADE_PATH}")
print(f"✓ ARCADE exists: {ARCADE_PATH.exists()}")

## Part 1: Mount Drive (Colab Only)

In [ ]:
# Only for Colab - Skip if running locally
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except:
    print("✓ Running locally (not Colab)")

## Part 2: Install Dependencies

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    'torch',
    'torchvision',
    'numpy',
    'pandas',
    'pillow',
    'matplotlib',
    'tqdm',
    'scikit-learn'
]

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

## Part 3: Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
from pathlib import Path
import json
from PIL import Image, ImageDraw
from collections import defaultdict

import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ PyTorch {torch.__version__}")
print(f"✓ Device: {device}")
print(f"✓ CUDA: {torch.cuda.is_available()}")

In [ ]:
# ===== TRAINING CONFIGURATION (Change these values) =====
EPOCHS = 20              # Number of training epochs
BATCH_SIZE = 4           # Batch size
LEARNING_RATE = 1e-3    # Learning rate
IMAGE_SIZE = 256        # Input image size
# =====================================================

print(f'Configuration:')
print(f'  EPOCHS: {EPOCHS}')
print(f'  BATCH_SIZE: {BATCH_SIZE}')
print(f'  LEARNING_RATE: {LEARNING_RATE}')
print(f'  IMAGE_SIZE: {IMAGE_SIZE}x{IMAGE_SIZE}')


In [ ]:
# ===== TRAINING CONFIGURATION (Change these values) =====
EPOCHS = 20              # Number of training epochs
BATCH_SIZE = 4           # Batch size
LEARNING_RATE = 1e-3    # Learning rate
IMAGE_SIZE = 256        # Input image size
# =====================================================

print(f'Configuration:')
print(f'  EPOCHS: {EPOCHS}')
print(f'  BATCH_SIZE: {BATCH_SIZE}')
print(f'  LEARNING_RATE: {LEARNING_RATE}')
print(f'  IMAGE_SIZE: {IMAGE_SIZE}x{IMAGE_SIZE}')


## Part 4: Load ARCADE Dataset

In [ ]:
# Load COCO annotations
train_ann_file = ARCADE_PATH / 'stenosis' / 'train' / 'annotations' / 'train.json'
val_ann_file = ARCADE_PATH / 'stenosis' / 'val' / 'annotations' / 'val.json'

with open(train_ann_file) as f:
    train_coco = json.load(f)
with open(val_ann_file) as f:
    val_coco = json.load(f)

TRAIN_IMAGES = ARCADE_PATH / 'stenosis' / 'train' / 'images'
VAL_IMAGES = ARCADE_PATH / 'stenosis' / 'val' / 'images'

print(f"✓ Train: {len(train_coco['images'])} images")
print(f"✓ Val: {len(val_coco['images'])} images")

## Part 5: Create Binary Vessel Masks

In [ ]:
def create_binary_mask(coco_data, image_info, w, h):
    """Convert COCO polygons to binary vessel mask."""
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    
    img_id = image_info['id']
    anns = [a for a in coco_data['annotations'] if a['image_id'] == img_id]
    
    for ann in anns:
        if 'segmentation' in ann and ann['segmentation']:
            for seg in ann['segmentation']:
                if len(seg) >= 6:
                    pts = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                    draw.polygon(pts, fill=255)
    
    return np.array(mask, dtype=np.uint8)

# Create masks
print("Creating masks...")
train_masks, val_masks = {}, {}

for img_info in tqdm(train_coco['images'], desc="Train"):
    img_id = img_info['id']
    w, h = img_info['width'], img_info['height']
    train_masks[img_id] = create_binary_mask(train_coco, img_info, w, h)

for img_info in tqdm(val_coco['images'], desc="Val"):
    img_id = img_info['id']
    w, h = img_info['width'], img_info['height']
    val_masks[img_id] = create_binary_mask(val_coco, img_info, w, h)

print(f"✓ Created {len(train_masks)} train masks")
print(f"✓ Created {len(val_masks)} val masks")

## Part 6: Visualize Sample Data

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for idx in range(3):
    img_info = train_coco['images'][idx]
    img_file = img_info['file_name']
    
    img = Image.open(TRAIN_IMAGES / img_file).convert('L')
    img_array = np.array(img)
    mask = train_masks[img_info['id']]
    
    axes[idx, 0].imshow(img_array, cmap='gray')
    axes[idx, 0].set_title(f"Image {idx+1}")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(mask, cmap='gray')
    axes[idx, 1].set_title(f"Mask {idx+1}")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'sample_data.png', dpi=100, bbox_inches='tight')
print("✓ Saved sample_data.png")
plt.show()

## Part 7: PyTorch Dataset Class

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, coco_data, image_dir, masks_dict, img_size=256):
        self.coco_data = coco_data
        self.image_dir = Path(image_dir)
        self.masks_dict = masks_dict
        self.img_size = img_size
        self.images = coco_data['images']
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        img_file = img_info['file_name']
        
        # Load image
        image = Image.open(self.image_dir / img_file).convert('L')
        image = image.resize((self.img_size, self.img_size), Image.BILINEAR)
        image = np.array(image, dtype=np.float32) / 255.0
        
        # Load mask
        mask = self.masks_dict[img_id]
        mask = Image.fromarray(mask).resize((self.img_size, self.img_size), Image.NEAREST)
        mask = np.array(mask, dtype=np.float32) / 255.0
        
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        
        return image, mask

train_dataset = SegmentationDataset(train_coco, TRAIN_IMAGES, train_masks, img_size=256)
val_dataset = SegmentationDataset(val_coco, VAL_IMAGES, val_masks, img_size=256)

print(f"✓ Datasets created: {len(train_dataset)} train, {len(val_dataset)} val")

## Part 8: Simple U-Net Architecture

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.enc1 = self._conv_block(1, 64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.enc2 = self._conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.enc3 = self._conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.bottleneck = self._conv_block(256, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = self._conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = self._conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = self._conv_block(128, 64)
        
        self.final = nn.Conv2d(64, 1, 1)
    
    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        enc1 = self.enc1(x)
        x = self.pool1(enc1)
        enc2 = self.enc2(x)
        x = self.pool2(enc2)
        enc3 = self.enc3(x)
        x = self.pool3(enc3)
        
        x = self.bottleneck(x)
        
        x = self.upconv3(x)
        x = torch.cat([x, enc3], dim=1)
        x = self.dec3(x)
        x = self.upconv2(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.dec2(x)
        x = self.upconv1(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.dec1(x)
        
        x = self.final(x)
        return x

model = SimpleUNet().to(device)
params = sum(p.numel() for p in model.parameters())
print(f"✓ U-Net created: {params:,} parameters")

## Part 9: Dice Loss

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        
        intersection = (pred * target).sum()
        dice = (2.0 * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1.0 - dice

criterion = DiceLoss()
print("✓ Dice Loss created")

## Part 10: Train the Model

In [ ]:
BATCH_SIZE = 4
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Training config: {BATCH_SIZE} batch, {NUM_EPOCHS} epochs, LR={LEARNING_RATE}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
train_losses, val_losses, val_dices = [], [], []
best_val_loss = float('inf')
best_epoch = 0

print(f"\n{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Val Dice':<12}")
print("="*50)

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Val
    model.eval()
    val_loss = 0.0
    dice_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
            pred_binary = (torch.sigmoid(outputs) > 0.5).float()
            intersection = (pred_binary * masks).sum()
            dice = (2 * intersection) / (pred_binary.sum() + masks.sum() + 1e-7)
            dice_scores.append(dice.item())
    
    val_loss /= len(val_loader)
    avg_dice = np.mean(dice_scores)
    val_losses.append(val_loss)
    val_dices.append(avg_dice)
    
    status = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        status = "✓ BEST"
        torch.save(model.state_dict(), RESULTS_PATH / 'best_unet.pth')
    
    print(f"{epoch+1:<8} {train_loss:<12.4f} {val_loss:<12.4f} {avg_dice:<12.4f} {status}")

print("="*50)
print(f"✓ Training complete! Best at epoch {best_epoch+1}")

## Part 11: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train', marker='o')
axes[0].plot(val_losses, label='Val', marker='s')
axes[0].axvline(best_epoch, color='r', linestyle='--', label='Best')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(val_dices, label='Validation Dice', marker='o', color='green')
axes[1].axvline(best_epoch, color='r', linestyle='--', label='Best')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Validation Dice')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'training_history.png', dpi=100, bbox_inches='tight')
print("✓ Saved training_history.png")
plt.show()

## Part 12: Final Evaluation

In [ ]:
model.load_state_dict(torch.load(RESULTS_PATH / 'best_unet.pth'))
model.eval()

all_dice = []
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        pred_binary = (torch.sigmoid(outputs) > 0.5).float()
        intersection = (pred_binary * masks).sum()
        dice = (2 * intersection) / (pred_binary.sum() + masks.sum() + 1e-7)
        all_dice.append(dice.item())

print(f"\n{'='*40}")
print(f"Final Results:")
print(f"{'='*40}")
print(f"Mean Dice:    {np.mean(all_dice):.4f}")
print(f"Std Dev:      {np.std(all_dice):.4f}")
print(f"Min Dice:     {np.min(all_dice):.4f}")
print(f"Max Dice:     {np.max(all_dice):.4f}")
print(f"Best Epoch:   {best_epoch+1}")
print(f"Results saved: {RESULTS_PATH}")

## Part 13: Visualize Predictions

In [ ]:
test_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
images, masks = next(iter(test_loader))
images, masks = images.to(device), masks.to(device)

with torch.no_grad():
    outputs = model(images)
    pred_probs = torch.sigmoid(outputs)
    pred_binary = (pred_probs > 0.5).float()

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i in range(4):
    axes[i, 0].imshow(images[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 0].set_title(f'Image {i+1}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(masks[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 1].set_title(f'Ground Truth')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred_probs[i, 0].cpu().numpy(), cmap='hot')
    axes[i, 2].set_title(f'Probability')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(pred_binary[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 3].set_title(f'Prediction')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'predictions.png', dpi=100, bbox_inches='tight')
print("✓ Saved predictions.png")
plt.show()

## Summary

✅ You've learned:
- How to load medical imaging datasets
- How U-Net architecture works
- What Dice loss measures
- How to train a segmentation model
- How to evaluate performance

📊 Results saved to: `results/1_simple_unet/`

🚀 Next: Try FPN, cGAN, or YOLOv8 models!

In [ ]:
print("\n" + "="*50)
print("✅ SEGMENTATION TUTORIAL COMPLETE!")
print("="*50)
print(f"\nMean Dice Score: {np.mean(all_dice):.4f}")
print(f"\nYou now understand:")
print(f"  ✓ Segmentation task")
print(f"  ✓ U-Net architecture")
print(f"  ✓ Training process")
print(f"  ✓ Evaluation metrics")
print(f"\n🎓 Ready to try advanced models!")